In [12]:
import os
import glob
import re
import pandas as pd
import numpy as np
import aqi
from sqlalchemy import create_engine
from dotenv import load_dotenv

In [13]:
DATA_DIR = "data/air/volvi/"
load_dotenv() 
DB_URL = os.getenv("DB_URL")

NAME_NORMALIZATION = {
    "Bolbes": "Volvi",
}

In [14]:
def calculate_hourly_aqi(row):
    try:
        sub_indices = []
        
        #Helper function to check both old ('o3') and new ('o3_conc') column names
        def get_val(base_name):
            val = row.get(f"{base_name}_conc")
            if pd.isna(val):
                val = row.get(base_name)
            return float(val) if pd.notna(val) else None

        pm25 = get_val('pm25')
        if pm25 is not None:
            sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_PM25, str(pm25)))
            
        pm10 = get_val('pm10')
        if pm10 is not None:
            sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_PM10, str(pm10)))
            
        no2 = get_val('no2')
        if no2 is not None:
            no2_ppb = (no2 * 24.45) / 46.01
            sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_NO2_1H, str(no2_ppb))) 
            
        co = get_val('co')
        if co is not None:
            co_ppm = (co * 24.45) / (28.01 * 1000)
            sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_CO_8H, str(co_ppm)))
            
        o3 = get_val('o3')
        if o3 is not None:
            o3_ppm = (o3 * 24.45) / (48.00 * 1000)
            sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_O3_8H, str(o3_ppm)))
            
        so2 = get_val('so2')
        if so2 is not None:
            so2_ppb = (so2 * 24.45) / 64.06
            sub_indices.append(aqi.to_iaqi(aqi.POLLUTANT_SO2_1H, str(so2_ppb)))
            
        return max(sub_indices) if sub_indices else np.nan
        
    except Exception as e:
        return np.nan

In [ ]:
file_pattern = os.path.join(DATA_DIR, "municipality_of_*_pollutants_conc_timeseries-yearly_*.csv")
all_files = glob.glob(file_pattern)

if not all_files:
    print(f"No CSV files found in '{DATA_DIR}/' folder!")

all_monthly_summaries = []
print(f"Found {len(all_files)} datasets. Beginning batch processing...\n")

for file_path in all_files:
    filename = os.path.basename(file_path)
    
    match = re.search(r'municipality_of_(.+?)_pollutants_conc_timeseries-yearly_(\d{4})', filename)
    if not match:
        print(f"Skipping unrecognized file format: {filename}")
        continue
        
    municipality_raw = match.group(1).replace("_", " ").title()
    #Applies normalization (e.g., Bolbes -> Volvi)
    municipality = NAME_NORMALIZATION.get(municipality_raw, municipality_raw)
    year_str = match.group(2)
    
    
    df = pd.read_csv(file_path)
    df['date'] = pd.to_datetime(df['time'])
    
    #Spatial aggregation (Handles older 2017-2021 grid data)
    if 'lat' in df.columns and 'lon' in df.columns:
        #Selects only numeric columns to average, plus the date column for grouping
        numeric_cols = df.select_dtypes(include='number').columns
        df = df.groupby('date')[numeric_cols].mean().reset_index()
    
    df['hourly_aqi'] = df.apply(calculate_hourly_aqi, axis=1)
    
    monthly_summary = df.groupby([df['date'].dt.year.rename('year'), 
                                  df['date'].dt.month.rename('month')])['hourly_aqi'].mean().reset_index()
    monthly_summary['mean_aqi'] = monthly_summary['hourly_aqi'].round(0)
    monthly_summary['municipality'] = municipality
    
    final_df = monthly_summary[['municipality', 'year', 'month', 'mean_aqi']]
    final_df = final_df.dropna(subset=['mean_aqi'])
    
    all_monthly_summaries.append(final_df)

Found 8 datasets. Beginning batch processing...

Processing Volvi (2017)...
   -> Spatial grid detected. Aggregating 394200 rows down to hourly averages...
Processing Volvi (2018)...
   -> Spatial grid detected. Aggregating 394200 rows down to hourly averages...
Processing Volvi (2019)...
   -> Spatial grid detected. Aggregating 525600 rows down to hourly averages...
Processing Volvi (2020)...
   -> Spatial grid detected. Aggregating 772992 rows down to hourly averages...
Processing Volvi (2021)...
   -> Spatial grid detected. Aggregating 770880 rows down to hourly averages...
Processing Volvi (2022)...
Processing Volvi (2023)...
Processing Volvi (2024)...


In [ ]:
if all_monthly_summaries:
    master_df = pd.concat(all_monthly_summaries, ignore_index=True)



Merging all datasets...
Total historical months calculated: 96
   municipality  year  month mean_aqi
0         Volvi  2017      1     25.0
1         Volvi  2017      2     29.0
2         Volvi  2017      3     34.0
3         Volvi  2017      4     38.0
4         Volvi  2017      5     38.0
5         Volvi  2017      6     39.0
6         Volvi  2017      7     39.0
7         Volvi  2017      8     41.0
8         Volvi  2017      9     35.0
9         Volvi  2017     10     29.0
10        Volvi  2017     11     23.0
11        Volvi  2017     12     21.0
12        Volvi  2018      1     22.0
13        Volvi  2018      2     25.0
14        Volvi  2018      3     32.0
15        Volvi  2018      4     35.0
16        Volvi  2018      5     35.0
17        Volvi  2018      6     37.0
18        Volvi  2018      7     37.0
19        Volvi  2018      8     37.0
20        Volvi  2018      9     36.0
21        Volvi  2018     10     31.0
22        Volvi  2018     11     23.0
23        Volvi  2018   

In [ ]:
engine = create_engine(DB_URL)
master_df.to_sql('historical_aqi', engine, if_exists='append', index=False)


Data successfully loaded into the database!
